# Chess Academy Pro — Voice Pack Generator

Generates HD voice packs using Kokoro TTS from **all** app phrases:
- 40 curated repertoire openings (overview, key ideas, traps, warnings, variation explanations)
- 1,903 walkthrough annotations (17,690 per-move narrations)
- 58 pro repertoire openings (Naroditsky, GothamChess, Carlsen, Hikaru, Caruana, Firouzja, Dubov)
- Generic drill hints and template messages

**Instructions:**
1. Click **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. When prompted, upload **three files** from your Mac:
   - `repertoire.json` → `~/Developer/chess-academy-pro/src/data`
   - `annotations-all.json` → `~/Developer/chess-academy-pro/src/data`
   - `pro-repertoires.json` → `~/Developer/chess-academy-pro/src/data`
4. Wait for generation to complete (~20-40 min per voice on T4)
5. Files auto-save to Google Drive + download as zip

**Tip:** To generate only Bella, set `VOICES_TO_GENERATE = ['af_bella']` in Cell 6.

In [ ]:
# Cell 1: Install dependencies + mount Google Drive
!pip install -q kokoro>=0.9.2 soundfile numpy
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

# Mount Google Drive so we don't lose work if runtime disconnects
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/ChessAcademyPro/voice_packs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Dependencies installed! Output will save to: {DRIVE_OUTPUT}')

In [ ]:
# Cell 2: Upload data files
import json, struct, os, time, zipfile
import numpy as np
import soundfile as sf
from io import BytesIO

print('Please upload THREE files:')
print('  1. repertoire.json          (40 curated openings)')
print('  2. annotations-all.json     (1,903 walkthrough annotations)')
print('  3. pro-repertoires.json     (58 pro player openings)')
print('')
print('Find all at: ~/Developer/chess-academy-pro/src/data')
print('')

from google.colab import files
uploaded = files.upload()

# Load files
repertoire = None
annotations_all = None
pro_repertoires = None

for fname, content in uploaded.items():
    data = json.loads(content.decode())
    if 'repertoire' in fname and 'pro' not in fname and 'annotation' not in fname:
        repertoire = data
        print(f'✓ Loaded {len(repertoire)} curated openings from {fname}')
    elif 'annotation' in fname:
        annotations_all = data
        total_moves = sum(len(moves) for moves in annotations_all.values())
        print(f'✓ Loaded {len(annotations_all)} annotation sets ({total_moves} moves) from {fname}')
    elif 'pro' in fname:
        pro_repertoires = data
        pro_count = len(data.get('openings', []))
        print(f'✓ Loaded {pro_count} pro repertoire openings from {fname}')

if not repertoire:
    raise FileNotFoundError('repertoire.json not uploaded!')
if not annotations_all:
    print('⚠ WARNING: annotations-all.json not uploaded — walkthrough annotations will be skipped')
if not pro_repertoires:
    print('⚠ WARNING: pro-repertoires.json not uploaded — pro openings will be skipped')

# Hash function — MUST match voicePackService.ts hashText() exactly
def hash_text(text: str) -> str:
    h = 0
    for ch in text:
        code = ord(ch)
        h = ((h << 5) - h) + code
        h &= 0xFFFFFFFF
        if h >= 0x80000000:
            h -= 0x100000000
    return str(h)

# Quick sanity check
assert hash_text('hello') == '99162322', f'Hash mismatch! Got {hash_text("hello")}'
print('\n✓ Hash function verified')

In [ ]:
# Cell 3: Collect all phrases from all sources
def collect_opening_phrases(opening):
    """Extract all speakable phrases from an opening dict."""
    phrases = set()
    if opening.get('overview'):
        phrases.add(opening['overview'])
    for idea in opening.get('keyIdeas', []):
        phrases.add(idea)
    for trap in opening.get('traps', []):
        phrases.add(trap)
    for warning in opening.get('warnings', []):
        phrases.add(warning)
    # Variation explanations + template messages
    for v in opening.get('variations', []):
        if v.get('explanation'):
            phrases.add(v['explanation'].replace('*', ''))
        phrases.add(f"Well done! You've completed the {v['name']} line.")
        phrases.add(f"Line discovered! You've learned the {v['name']}.")
        phrases.add(f"Line perfected! You know the {v['name']} by heart.")
    # Trap lines
    for t in opening.get('trapLines', []):
        if t.get('explanation'):
            phrases.add(t['explanation'].replace('*', ''))
        phrases.add(f"Well done! You've completed the {t['name']} line.")
        phrases.add(f"Line discovered! You've learned the {t['name']}.")
    # Warning lines
    for w in opening.get('warningLines', []):
        if w.get('explanation'):
            phrases.add(w['explanation'].replace('*', ''))
        phrases.add(f"Well done! You've completed the {w['name']} line.")
        phrases.add(f"Line discovered! You've learned the {w['name']}.")
    # Play mode intro
    phrases.add(f"Let's play the {opening['name']}. Remember your key ideas and play confidently.")
    return phrases

def collect_all_phrases(repertoire, annotations_all=None, pro_repertoires=None):
    phrases = set()
    
    # 1. Curated repertoire openings (40)
    for opening in repertoire:
        phrases |= collect_opening_phrases(opening)
    
    # 2. Walkthrough annotations (1,903 openings × ~9.3 moves each)
    if annotations_all:
        for opening_id, moves in annotations_all.items():
            for move in moves:
                if move.get('annotation'):
                    phrases.add(move['annotation'])
    
    # 3. Pro repertoire openings (58)
    if pro_repertoires:
        for opening in pro_repertoires.get('openings', []):
            phrases |= collect_opening_phrases(opening)
    
    # 4. Generic drill hints
    for hint in [
        'Castle to safety.', 'Develop your knight.', 'Develop your bishop.',
        'Bring your queen out.', 'Activate your rook.', 'Continue with the plan.',
    ]:
        phrases.add(hint)
    
    return sorted(phrases)

phrases = collect_all_phrases(repertoire, annotations_all, pro_repertoires)

# Stats breakdown
rep_phrases = set()
for o in repertoire:
    rep_phrases |= collect_opening_phrases(o)
anno_phrases = set()
if annotations_all:
    for moves in annotations_all.values():
        for m in moves:
            if m.get('annotation'):
                anno_phrases.add(m['annotation'])
pro_phrases = set()
if pro_repertoires:
    for o in pro_repertoires.get('openings', []):
        pro_phrases |= collect_opening_phrases(o)

print(f'Collected {len(phrases)} unique phrases:')
print(f'  Repertoire openings:     {len(rep_phrases)}')
print(f'  Walkthrough annotations: {len(anno_phrases)}')
print(f'  Pro repertoire openings: {len(pro_phrases)}')
print(f'  Generic hints:           6')
print(f'  (some overlap removed by dedup)')

In [ ]:
# Cell 4: Initialize Kokoro pipeline
from kokoro import KPipeline

# American English pipeline
pipeline_a = KPipeline(lang_code='a')
# British English pipeline
pipeline_b = KPipeline(lang_code='b')
print('Kokoro pipelines ready!')

In [ ]:
# Cell 5: Pack voice clips into binary format
# Format: [count:uint32] repeated [hashLen:uint16][hash:utf8][audioLen:uint32][mp3Data:bytes]

def audio_to_mp3_bytes(audio_np, sample_rate=24000):
    """Convert numpy audio array to MP3 bytes via WAV intermediary."""
    buf = BytesIO()
    sf.write(buf, audio_np, sample_rate, format='WAV')
    return buf.getvalue()  # WAV bytes (universally decodable)

def pack_voice_pack(clips):
    parts = [struct.pack('<I', len(clips))]
    for clip in clips:
        hash_bytes = clip['hash'].encode('utf-8')
        audio_data = clip['audio']
        parts.append(struct.pack('<H', len(hash_bytes)))
        parts.append(hash_bytes)
        parts.append(struct.pack('<I', len(audio_data)))
        parts.append(audio_data)
    return b''.join(parts)

print('Packing functions ready!')

In [ ]:
# Cell 6: Generate voice packs
# ⚡ To generate only Bella, change this to: VOICES_TO_GENERATE = ['af_bella']
VOICES_TO_GENERATE = ['af_bella']  # Start with just Bella to test the pipeline

ALL_VOICES = {
    'af_heart': 'a', 'af_bella': 'a', 'af_nicole': 'a', 'af_sarah': 'a', 'af_nova': 'a',
    'am_adam': 'a', 'am_eric': 'a', 'am_michael': 'a', 'am_liam': 'a',
    'bf_emma': 'b', 'bf_isabella': 'b',
    'bm_daniel': 'b', 'bm_george': 'b',
}

os.makedirs('voice_packs', exist_ok=True)

for voice_id in VOICES_TO_GENERATE:
    lang = ALL_VOICES[voice_id]
    pipeline = pipeline_a if lang == 'a' else pipeline_b
    print(f'\n{"="*60}')
    print(f'Generating: {voice_id} ({len(phrases)} phrases)')
    print(f'{"="*60}')
    
    clips = []
    start = time.time()
    errors = 0
    
    for i, text in enumerate(phrases):
        text_hash = hash_text(text)
        try:
            generator = pipeline(text, voice=voice_id, speed=1.0)
            all_audio = []
            for _, _, audio_chunk in generator:
                all_audio.append(audio_chunk)
            
            if all_audio:
                full_audio = np.concatenate(all_audio)
                wav_bytes = audio_to_mp3_bytes(full_audio, 24000)
                clips.append({'hash': text_hash, 'audio': wav_bytes})
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f'  ERROR [{i+1}]: {str(e)[:80]}')
        
        if (i + 1) % 200 == 0 or i == len(phrases) - 1:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed * 60
            eta = (len(phrases) - i - 1) / max(rate / 60, 0.01) / 60
            print(f'  [{i+1}/{len(phrases)}] {elapsed:.0f}s, {rate:.0f}/min, ETA {eta:.1f}min, {errors} errors')
        
        # Save checkpoint to Drive every 500 clips
        if (i + 1) % 500 == 0:
            checkpoint = pack_voice_pack(clips)
            cp_path = f'{DRIVE_OUTPUT}/{voice_id}_checkpoint.bin'
            with open(cp_path, 'wb') as f:
                f.write(checkpoint)
            print(f'  💾 Checkpoint saved ({len(clips)} clips)')
    
    # Pack and save locally + to Drive
    packed = pack_voice_pack(clips)
    local_path = f'voice_packs/{voice_id}.bin'
    drive_path = f'{DRIVE_OUTPUT}/{voice_id}.bin'
    
    with open(local_path, 'wb') as f:
        f.write(packed)
    with open(drive_path, 'wb') as f:
        f.write(packed)
    
    # Remove checkpoint now that final is saved
    cp_path = f'{DRIVE_OUTPUT}/{voice_id}_checkpoint.bin'
    if os.path.exists(cp_path):
        os.remove(cp_path)
    
    size_mb = len(packed) / 1024 / 1024
    print(f'\n  ✅ Saved: {local_path} ({size_mb:.1f} MB, {len(clips)} clips, {errors} errors)')
    print(f'  ✅ Backed up to Google Drive: {drive_path}')

print(f'\n{"="*60}')
print(f'All {len(VOICES_TO_GENERATE)} voice pack(s) generated!')
print(f'{"="*60}')

In [ ]:
# Cell 7: Zip and download (also saved on Google Drive)
zip_path = 'voice_packs.zip'
drive_zip = f'{DRIVE_OUTPUT}/voice_packs.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir('voice_packs')):
        if fname.endswith('.bin'):
            fpath = os.path.join('voice_packs', fname)
            zf.write(fpath, fname)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  Added {fname} ({size_mb:.1f} MB)')

# Copy zip to Drive
import shutil
shutil.copy2(zip_path, drive_zip)

zip_size = os.path.getsize(zip_path) / 1024 / 1024
print(f'\nZip: {zip_path} ({zip_size:.1f} MB)')
print(f'Drive backup: {drive_zip}')

# Auto-download
try:
    from google.colab import files
    files.download(zip_path)
    print('Download started!')
except:
    print(f'Not in Colab — file at {zip_path}')

print(f'\n📂 Your voice packs are also safe in Google Drive:')
print(f'   {DRIVE_OUTPUT}/')
print(f'   (won\'t be lost if runtime disconnects!)')